# Base Model vs Fine-tuned Model Inference

동일한 test context를 베이스 모델과 파인튜닝 모델에 입력하여 결과를 비교합니다.

In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
FastLanguageModel.for_inference(model)

In [ ]:
instruction = "Read the passage and generate exactly one meaningful quiz question that can be answered using only the passage. Output only the question."
context = "Paste an unseen test context here."
messages = [{"role": "user", "content": f"{instruction}\n\nPassage:\n{context}"}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=80, temperature=0.2, use_cache=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

파인튜닝 adapter를 불러온 뒤 동일한 context로 다시 추론하고, 결과를 `results/comparison.csv`에 기록합니다.